# SNN MNIST Classification with Rate Coding

Лабораторная работа: **Спайковые нейронные сети**  
Задача: классификация цифр MNIST с использованием **rate coding**.

Что реализовано:
- Загрузка и нормализация MNIST
- Кодирование пикселей в Poisson spike trains
- LIF нейроны
- Классификация 10 цифр
- Raster plot
- График мембранного потенциала


In [ ]:
!pip install torch torchvision snntorch matplotlib

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import snntorch as snn
from snntorch import spikeplot as splt
from snntorch import spikegen

import matplotlib.pyplot as plt
import numpy as np
import random

## Фиксируем random seed (воспроизводимость эксперимента)

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

## Параметры модели

In [ ]:
batch_size = 64
num_steps = 25
beta = 0.95
lr = 1e-3

## Загрузка и нормализация MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0,), (1,))
])

mnist_train = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# используем только часть датасета
subset_indices = torch.arange(0, 10000)
mnist_train = torch.utils.data.Subset(mnist_train, subset_indices)

train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)

## Архитектура SNN (LIF нейроны)

In [ ]:
num_inputs = 28*28
num_hidden = 256
num_outputs = 10

class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta)

        self.fc2 = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, x):

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        spk2_rec = []
        mem2_rec = []

        for step in range(num_steps):

            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            spk2_rec.append(spk2)
            mem2_rec.append(mem2)

        return torch.stack(spk2_rec), torch.stack(mem2_rec)

net = Net()

## Loss и Optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

## Обучение сети

In [ ]:
loss_hist = []

for epoch in range(1):

    for data, targets in train_loader:

        data = data.view(batch_size, -1)

        spike_data = spikegen.rate(data, num_steps=num_steps)

        optimizer.zero_grad()

        spk_rec, mem_rec = net(spike_data[0])

        spike_count = spk_rec.sum(0)

        loss = loss_fn(spike_count, targets)

        loss.backward()
        optimizer.step()

        loss_hist.append(loss.item())

print('Training loss:', np.mean(loss_hist))

## Raster Plot

In [ ]:
data, targets = next(iter(train_loader))
data = data.view(batch_size, -1)

spike_data = spikegen.rate(data, num_steps=num_steps)

spk_rec, mem_rec = net(spike_data[0])

plt.figure()
splt.raster(spk_rec[:,0].detach().cpu())
plt.title('Spike Raster Plot')
plt.show()

## Мембранный потенциал

In [ ]:
plt.figure()
plt.plot(mem_rec[:,0].detach().cpu())
plt.title('Membrane Potential')
plt.show()

## График функции потерь

In [ ]:
plt.figure()
plt.plot(loss_hist)
plt.title('Training Loss')
plt.show()